In [18]:
import os
import glob
import fitz
import re

DATA_DIR = os.path.expanduser("~/Desktop/projects-learning/PillWise/data/raw_pdfs")
pdf_paths = glob.glob(os.path.join(DATA_DIR, "**/*.pdf"), recursive=True)
pdf_paths.sort()

def extract_text(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text.strip()

def fixed_size_chunks(text, chunk_size=500, overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - overlap
    return chunks

documents = []
for path in pdf_paths:
    text = extract_text(path)
    drug_name = path.split("/")[-2]
    documents.append({"text": text, "source": path, "drug": drug_name})

all_chunks_fixed = []
for doc in documents:
    for chunk in fixed_size_chunks(doc['text']):
        all_chunks_fixed.append({
            "chunk": chunk,
            "drug": doc['drug'],
            "source": doc['source'],
            "strategy": "fixed"
        })

print(f"chunks ready: {len(all_chunks_fixed)}")

chunks ready: 3139


In [19]:
import os
from dotenv import load_dotenv
from google import genai

load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
client = genai.Client(api_key = GEMINI_API_KEY)

print(f"API key loaded: {GEMINI_API_KEY[:8]}...")

API key loaded: AQ.Ab8RN...


In [20]:
result = client.models.embed_content(
    model="gemini-embedding-001",
    contents="Acetaminophen is a pain reliever and fever reducer."
)

embedding = result.embeddings[0].values
print(f"embedding dimensions: {len(embedding)}")
print(f"first 5 values      : {embedding[:5]}")

embedding dimensions: 3072
first 5 values      : [-0.01036844, 0.001454646, 0.026839476, -0.049040943, 0.0028268993]


In [21]:
import time
def embed_chunks(chunks, batch_size = 10):
    embeddings = []
    total = len(chunks)

    for i in range(0, total, batch_size):
        batch = chunks[i:i + batch_size]
        texts = [c['chunk'] for c in batch]

        result = client.models.embed_content(
        model = "gemini-embedding-001",
        contents = texts
        )

        for j, embedding in enumerate(result.embeddings):
            embeddings.append({
                **batch[j],
                "embedding": embedding.values
            })
            
        print(f"embedded {min(i + batch_size, total)}/{total}")
        time.sleep(1)

    return embeddings

test_embeddings = embed_chunks(all_chunks_fixed[:20])
print(f"\ndone: {len(test_embeddings)} embeddings")
print(f"embedding size: {len(test_embeddings[0]['embedding'])}")

embedded 10/20
embedded 20/20

done: 20 embeddings
embedding size: 3072
